# Video embedding search with Qwen3-VL-Embedding and FAISS

This notebook searches a video by meaning. It cuts the video into short clips, turns each clip into an embedding vector with `Qwen/Qwen3-VL-Embedding-8B` (served by vLLM and reached over HTTP), stores the vectors in a local **FAISS** index, and finds the clips closest to a text or image query.

**Flow**

![Video search pipeline: indexing and query flow](../images/video-search-pipeline.svg)

Every vector is stored together with the **time range** of the clip it came from, so a search result is a timestamp you can jump to in the video.

Prerequisite: an embedding server must be reachable at `BASE_URL` (default `http://127.0.0.1:18001`), and for section 7 a reranker server at `RERANK_URL`. See the README for how to start both with vLLM. This notebook accompanies the blog post *Semantic Video Search for Media Archives*.

In [ ]:
%pip install -q numpy faiss-cpu opencv-python-headless pillow requests

In [ ]:
import os, time, json, base64, tempfile
from concurrent.futures import ThreadPoolExecutor

import cv2
import faiss
import numpy as np
import requests

BASE_URL = "http://127.0.0.1:18001"                  # embedding server (OpenAI-style API, e.g. vLLM)
MODEL = "Qwen/Qwen3-VL-Embedding-8B"

VIDEO_PATH = "data/meridian.mp4"   # downloaded ahead of time, see section 1

SEGMENT_SEC = 5          # length of each clip
ANALYZE_SEC = None       # only embed the first N seconds (None = whole video)
CLIP_WIDTH, CLIP_FPS = 448, 2   # clips are downscaled and sampled at 2 fps: the model context is only 8192 tokens
KEYFRAMES_PER_CLIP = 3   # still frames also embedded per clip, used for image queries
WORKERS = 4              # parallel embedding requests
TOP_K = 5

RERANK_URL = "http://127.0.0.1:18000/v1"            # reranker server (vLLM)
RERANK_MODEL = "Qwen/Qwen3-VL-Reranker-2B"
RUN_RERANK = True       # set False to skip section 7 if you have no reranker server
CANDIDATES = 12          # clips from embedding search that the reranker re-scores

info = requests.get(f"{BASE_URL}/v1/models", timeout=10).json()["data"][0]
print("model server:", info["id"], "| max_model_len:", info["max_model_len"])

## 1. Load the video

The video is **Meridian** (2016), a short film directed by Curtis Clark and released by Netflix and Xiph.org (about 12 minutes, 720p, with audio, about 49 MB). It has a good mix of content to search: a 1940s city street, a detective in an office, a car interior, police cars on a coastal bluff, a stormy beach, a man with a flashlight in the dark, and end credits with on-screen text.

> **License.** Meridian is released under the Creative Commons Attribution-NonCommercial-NoDerivatives 4.0 International license, as listed on its [Wikipedia page](https://en.wikipedia.org/wiki/Meridian_(film)). Check the terms at the source before reusing the footage. The video is not stored in this repository.

**Get the video**

1. Download Meridian from the source you trust (Netflix Open Content or Xiph.org).
2. Save it as `data/meridian.mp4` in this folder:
   ```bash
   mkdir -p data
   mv ~/Downloads/<downloaded file>.mp4 data/meridian.mp4
   ```

The cell below checks that `VIDEO_PATH` exists and prints its size and length. Below the code, a strip of frames sampled across the video shows what it contains.

To use another video, put the file in `data/` and change `VIDEO_PATH`.


In [ ]:
assert os.path.exists(VIDEO_PATH), f"{VIDEO_PATH} not found: download the video first (see the instructions above)"

cap = cv2.VideoCapture(VIDEO_PATH)
SRC_FPS = cap.get(cv2.CAP_PROP_FPS)
N_FRAMES = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
DURATION = N_FRAMES / SRC_FPS
cap.release()
print(f"{VIDEO_PATH}: {os.path.getsize(VIDEO_PATH) / 1e6:.1f} MB, {DURATION:.0f} s, {SRC_FPS:.2f} fps")

# Preview: a strip of evenly spaced frames (small JPEGs, so the notebook stays light; the original file is what gets analysed).
from IPython.display import HTML, display

PREVIEW_FRAMES = 6
cap = cv2.VideoCapture(VIDEO_PATH)
tiles = ""
for n in range(PREVIEW_FRAMES):
    sec = (n + 0.5) * DURATION / PREVIEW_FRAMES
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(sec * SRC_FPS))
    ok, frame = cap.read()
    if ok:
        frame = cv2.resize(frame, (320, int(frame.shape[0] * 320 / frame.shape[1])))
        b64 = base64.b64encode(cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, 70])[1]).decode()
        tiles += (f"<div style='width:320px'><img src='data:image/jpeg;base64,{b64}' style='width:100%;border-radius:6px'>"
                  f"<div style='font:12px sans-serif;color:#57606a'>{int(sec) // 60}:{int(sec) % 60:02d}</div></div>")
cap.release()
display(HTML(f"<div style='display:flex;flex-wrap:wrap;gap:10px'>{tiles}</div>"))


## 2. Cut the video into clips

Qwen3-VL-Embedding embeds whatever you send it as one input, so we cut the video ourselves: fixed `SEGMENT_SEC` windows, each with `startSec` and `endSec`. Each clip is decoded, downscaled to `CLIP_WIDTH` pixels wide, sampled at `CLIP_FPS` and written as a small MP4. A clip is about 50 KB and about 700 tokens for the model.

In [ ]:
def mmss(sec):
    return f"{int(sec // 60):02d}:{int(sec % 60):02d}"

end_sec = min(DURATION, ANALYZE_SEC) if ANALYZE_SEC else DURATION
segments = []
t = 0.0
while t < end_sec - 1:                                  # drop a final sliver shorter than 1 s
    segments.append({"row": len(segments), "startSec": t, "endSec": min(t + SEGMENT_SEC, end_sec)})
    t += SEGMENT_SEC
print(len(segments), "clips of", SEGMENT_SEC, "s")

def read_frames(start, end, width=CLIP_WIDTH, fps=CLIP_FPS):
    """Decode [start, end) seconds, keep `fps` frames per second, resize to `width`."""
    cap = cv2.VideoCapture(VIDEO_PATH)                  # one capture per call, so threads don't share state
    first, last, step = int(start * SRC_FPS), int(end * SRC_FPS), max(int(SRC_FPS / fps), 1)
    cap.set(cv2.CAP_PROP_POS_FRAMES, first)
    frames = []
    for i in range(first, last):
        ok, frame = cap.read()
        if not ok:
            break
        if (i - first) % step == 0:
            frames.append(cv2.resize(frame, (width, int(frame.shape[0] * width / frame.shape[1]))))
    cap.release()
    return frames

def encode_clip(frames, fps=CLIP_FPS):
    path = tempfile.mktemp(suffix=".mp4")
    h, w = frames[0].shape[:2]
    writer = cv2.VideoWriter(path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (w, h))
    for f in frames:
        writer.write(f)
    writer.release()
    data = open(path, "rb").read()
    os.remove(path)
    return data

def jpeg_b64(frame, quality=80):
    return base64.b64encode(cv2.imencode(".jpg", frame, [cv2.IMWRITE_JPEG_QUALITY, quality])[1].tobytes()).decode()

def read_frame_at(sec, width=CLIP_WIDTH):
    cap = cv2.VideoCapture(VIDEO_PATH)
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(min(sec, DURATION - 0.5) * SRC_FPS))
    ok, frame = cap.read()
    cap.release()
    return cv2.resize(frame, (width, int(frame.shape[0] * width / frame.shape[1])))

demo = read_frames(60, 65)
print(f"example clip 01:00-01:05: {len(demo)} frames of {demo[0].shape[1]}x{demo[0].shape[0]}, {len(encode_clip(demo)) / 1e3:.0f} KB")

## 3. Embed the clips with Qwen3-VL-Embedding

vLLM serves the model through the OpenAI-style `/v1/embeddings` endpoint. For multimodal input it takes a chat-style `messages` list; each part is `text`, `image_url` or `video_url` (base64 data URLs here). The response is one vector per request. The same model embeds text, images and video into **one shared space**, which is what makes text → video search possible.

Images and video go in the `messages` form. A plain text query is sent as a simple `input` string instead (see the comment in the code).

Vectors are L2-normalised so the inner product equals cosine similarity.

We embed **two kinds of vectors**:
- one **clip vector** per clip (the video itself), used for *text* queries;
- a few **keyframe vectors** per clip (still frames, `KEYFRAMES_PER_CLIP`), used for *image* queries.

Why both? In testing, this model compares an image with another image far more reliably than an image with a video clip, while a text query matches video clips well and still frames poorly. Matching each query type with the vectors it works best against gives better results than one index for everything.

In [ ]:
def text_part(text):
    return {"type": "text", "text": text}

def image_part(frame):
    return {"type": "image_url", "image_url": {"url": "data:image/jpeg;base64," + jpeg_b64(frame)}}

def video_part(mp4_bytes):
    return {"type": "video_url", "video_url": {"url": "data:video/mp4;base64," + base64.b64encode(mp4_bytes).decode()}}

def embed(parts):
    """Embed one multimodal input (a list of text/image/video parts) -> normalised float32 vector."""
    if all(p["type"] == "text" for p in parts):
        # Text-only queries use the plain `input` form. In a quick test it retrieved better
        # than the chat-style `messages` form (more of the top-5 clips were correct).
        body = {"model": MODEL, "encoding_format": "float", "input": " ".join(p["text"] for p in parts)}
    else:
        body = {"model": MODEL, "encoding_format": "float", "messages": [{"role": "user", "content": parts}]}
    r = requests.post(f"{BASE_URL}/v1/embeddings", timeout=300, json=body)
    if r.status_code != 200:
        raise RuntimeError(f"{r.status_code}: {r.text[:300]}")
    v = np.asarray(r.json()["data"][0]["embedding"], dtype="float32")
    return v / np.linalg.norm(v)

timings = {}
def timed(step, detail, fn):
    t0 = time.perf_counter()
    out = fn()
    timings[step] = (time.perf_counter() - t0, detail)
    print(f"⏱ {step}: {timings[step][0]:.2f} s  ({detail})")
    return out

def embed_segment(seg):
    frames = read_frames(seg["startSec"], seg["endSec"])
    seg["thumb"] = jpeg_b64(frames[len(frames) // 2], 70)          # keep a thumbnail for the results page
    return embed([video_part(encode_clip(frames))])

def embed_all():
    with ThreadPoolExecutor(WORKERS) as pool:
        return list(pool.map(embed_segment, segments))

E = np.vstack(timed("Embed clips", f"{len(segments)} clips, {WORKERS} parallel requests", embed_all))
print("clip embedding matrix:", E.shape)

# Still frames: a text query is matched against the clip vectors, an image query against these frame vectors
keyframes = [{"row": s["row"], "sec": s["startSec"] + (k + 0.5) / KEYFRAMES_PER_CLIP * (s["endSec"] - s["startSec"])}
             for s in segments for k in range(KEYFRAMES_PER_CLIP)]

def embed_keyframe(kf):
    return embed([image_part(read_frame_at(kf["sec"]))])

def embed_keyframes():
    with ThreadPoolExecutor(WORKERS) as pool:
        return list(pool.map(embed_keyframe, keyframes))

F = np.vstack(timed("Embed keyframes", f"{len(keyframes)} frames, {WORKERS} parallel requests", embed_keyframes))
print("keyframe embedding matrix:", F.shape)

## 4. Store the vectors in FAISS

Two flat inner-product indexes do an exact search: `index` for the clip vectors and `frame_index` for the keyframe vectors; for a video of this size that is instant. FAISS returns **row numbers**. Row *i* of `index` is `segments[i]`, and each row of `frame_index` is `keyframes[j]`, which records the clip row it belongs to: that is how an embedding maps back to a `startSec` / `endSec`. The index and the segment metadata are also saved to disk so the notebook can reload them without re-embedding.

In [ ]:
index = faiss.IndexFlatIP(E.shape[1])           # clip vectors, used for text queries
index.add(E)
frame_index = faiss.IndexFlatIP(F.shape[1])      # keyframe vectors, used for image queries
frame_index.add(F)

meta = [{k: v for k, v in s.items() if k != "thumb"} for s in segments]
faiss.write_index(index, "video.faiss")
faiss.write_index(frame_index, "video_frames.faiss")
json.dump({"segments": meta, "keyframes": keyframes}, open("video_segments.json", "w"))

print("clip index:", index.ntotal, "vectors | keyframe index:", frame_index.ntotal, "vectors | dim:", index.d)
print("clip row 0 ->", meta[0], "| first 5 dims:", np.round(E[0, :5], 4))
print("keyframe row 0 ->", keyframes[0], "(belongs to clip row", keyframes[0]["row"], ")")

## 5. Search with text and with an image

The query goes through the **same model** and the same normalisation, then FAISS returns the `TOP_K` nearest clips.

- **Text query** → compared with the **clip vectors** (`search_clips`).
- **Image query** → compared with the **keyframe vectors** (`search_frames`); the best frame of each clip sets that clip's score. Here we use a frame taken from the video as the query picture.

Cosine scores between text and video are much lower than between two images (a known gap between modalities). Compare scores **within one query**, not across queries.

An image query finds clips that *look the same or similar*. The clip the frame came from usually ranks first or second; neighbouring clips of the same shot score almost as high, because consecutive seconds of a scene look alike.

In [ ]:
def search_clips(parts, k=TOP_K):
    """Text query -> nearest clip vectors."""
    q = embed(parts)
    scores, rows = index.search(q[None, :], k)
    return q, [{"rank": n + 1, "score": float(s), **segments[r]} for n, (s, r) in enumerate(zip(scores[0], rows[0]))]

def search_frames(parts, k=TOP_K):
    """Image query -> nearest keyframe vectors, then the best frame of each clip decides that clip's score."""
    q = embed(parts)
    scores, ids = frame_index.search(q[None, :], min(frame_index.ntotal, k * KEYFRAMES_PER_CLIP * 4))
    best = {}
    for s, i in zip(scores[0], ids[0]):
        row = keyframes[i]["row"]
        best.setdefault(row, float(s))           # results come sorted, so the first hit per clip is its best frame
    top = list(best.items())[:k]
    return q, [{"rank": n + 1, "score": s, **segments[row]} for n, (row, s) in enumerate(top)]

text_query = "two vintage police cars parked by the ocean"
q_text, text_hits = timed("Text query + search", text_query, lambda: search_clips([text_part(text_query)]))

image_query_sec = 314.0                  # a frame from the video, as the query image
query_frame = read_frame_at(image_query_sec)
q_image, image_hits = timed("Image query + search", f"frame at {mmss(image_query_sec)}", lambda: search_frames([image_part(query_frame)]))

## 6. Show the matching clips

Each result is a clip with its time range and similarity. Each result shows a thumbnail of the clip and its `mm:ss` range, so it works in any notebook front end. Open the video at that time to watch the clip.

In [ ]:
from IPython.display import HTML, display
import html as _html

CSS = """
<style>
.vr {display:flex; gap:12px; flex-wrap:wrap; font:13px sans-serif; color:#1f2328}
.vc {border:1px solid #d0d7de; border-radius:8px; padding:8px; background:#fff; width:230px}
.vc img {width:100%; border-radius:4px; display:block}
.vc b {font-size:15px} .rk {background:#0550ae; color:#fff; border-radius:10px; padding:1px 8px; margin-right:6px; font-size:12px}
table.t {border-collapse:collapse; font:13px sans-serif; color:#1f2328} table.t th,table.t td {border:1px solid #d0d7de; padding:5px 9px; text-align:left}
table.t th {background:#f6f8fa} .num {text-align:right; font-family:monospace}
</style>
"""

def show_hits(title, hits, query_html="", score_label="cosine"):
    cards = "".join(
        f"<div class='vc'><img src='data:image/jpeg;base64,{h['thumb']}'><div style='margin-top:6px'>"
        f"<span class='rk'>#{h['rank']}</span><b>{mmss(h['startSec'])} – {mmss(h['endSec'])}</b></div>"
        f"<div>{score_label} {h['score']:.3f} · row {h['row']}" + (f" · was #{h['was']}" if "was" in h else "") + "</div></div>" for h in hits)
    display(HTML(CSS + f"<h4 style='font:600 14px sans-serif'>{_html.escape(title)}</h4>{query_html}<div class='vr'>{cards}</div>"))

show_hits(f"Text query: “{text_query}”", text_hits)
show_hits(f"Image query: frame at {mmss(image_query_sec)}", image_hits,
          f"<img style='width:230px;border-radius:4px;margin-bottom:8px' src='data:image/jpeg;base64,{jpeg_b64(query_frame, 70)}'>")

## 7. Rerank the candidates

Embedding search compares two vectors that never see each other. A **reranker** reads the query and a candidate **together** and scores how well they match, so it is better at fine detail: exact on-screen words, or telling one scene from a similar one. It is slower, so it only re-scores a shortlist: the top `CANDIDATES` clips from embedding search.

The reranker here is `Qwen/Qwen3-VL-Reranker-2B`, a vision-language model served by vLLM and reached at `RERANK_URL` (see the README). It takes a text query and **image** passages, not video, so each candidate clip is scored through its keyframes, and each frame is scored in its own request, and **a clip's score is its best frame's score**. Set `RUN_RERANK = False` in the config cell to skip this section if you have no reranker server.

Reading the results: scores are relevance probabilities between 0 and 1, higher is more relevant. Near-identical frames (the same slide in neighbouring seconds) get almost the same score, so their order is arbitrary. Reranking helps most when the query is specific; when embedding search is already right, the order barely changes.

In [ ]:
def rerank_clips(query, candidates):
    """Re-score candidate clips with the VL reranker; each clip is scored by its best keyframe."""
    frames, owners = [], []
    for c in candidates:
        for kf in (k for k in keyframes if k["row"] == c["row"]):
            frames.append("data:image/jpeg;base64," + jpeg_b64(read_frame_at(kf["sec"])))
            owners.append(c["row"])

    def score(image_url):
        r = requests.post(f"{RERANK_URL}/rerank", timeout=300,
                          json={"model": RERANK_MODEL, "query": query,
                                "documents": {"content": [{"type": "image_url", "image_url": {"url": image_url}}]}})
        if r.status_code != 200:
            raise RuntimeError(f"{r.status_code}: {r.text[:300]}")
        return r.json()["results"][0]["relevance_score"]

    with ThreadPoolExecutor(WORKERS) as pool:
        scores = list(pool.map(score, frames))
    best = {}
    for row, sc in zip(owners, scores):
        best[row] = max(best.get(row, -1e9), sc)
    ranked = sorted(candidates, key=lambda c: -best[c["row"]])
    return [{**c, "was": c["rank"], "rank": n + 1, "score": best[c["row"]]} for n, c in enumerate(ranked)], len(frames)

if RUN_RERANK:
    for rerank_query in ["a man in a hat holding a flashlight in the dark", "a man standing alone on a rocky beach"]:
        _, candidates = search_clips([text_part(rerank_query)], k=CANDIDATES)
        reranked, _ = timed(f"Rerank: {rerank_query[:28]}", f"{len(candidates)} clips, {len(candidates) * KEYFRAMES_PER_CLIP} frames",
                                   lambda: rerank_clips(rerank_query, candidates))
        show_hits(f"Embedding search, top {TOP_K}: “{rerank_query}”", candidates[:TOP_K])
        show_hits(f"After reranking, top {TOP_K}", reranked[:TOP_K], score_label="rerank score")

## 8. Search text that appears on screen (hard captions)

Burned-in text, such as a title card, a sign or a hard-coded subtitle, is part of the picture, so the model can read it from the frames. A text query is compared with the **keyframe vectors**, so it finds the shots where those words are on screen. The sample video's end credits are a good test, for example "Visual Effects Supervision by Bill Taylor, ASC".

In the sample run, the top result is the credit card with exactly that line. The second result is another credit card with the same layout but different words ("Original Music by ..."), so the model matches the look of a credit card as well as the letters. Treat the top matches as candidates and check the thumbnails.

The second cell checks that the model really reads the letters: it burns an invented caption onto a frame that has no text and compares the query's similarity with and without the caption.

**What this model cannot search**
- **Spoken audio / transcript:** the model takes text, images and video frames only; the endpoint rejects audio input. To search dialogue, transcribe the audio with a speech-recognition model, embed each transcript segment as text with this same model, and add it to the index with its start and end time.
- **Music and sound:** this needs a separate audio-embedding model; Qwen3-VL-Embedding cannot hear it.
- **Small or stylised text:** reading is reliable for large, clear text and weaker for small, blurry or heavily stylised text.

In [ ]:
caption_query = "Visual Effects Supervision by Bill Taylor, ASC"   # end-credit text shown on screen
q_cap, cap_hits = timed("Caption query + search", caption_query, lambda: search_frames([text_part(caption_query)]))
show_hits(f"On-screen text query: “{caption_query}”", cap_hits)

In [ ]:
# Does the model read the letters? Burn an invented caption onto a frame that has no text.
plain = read_frame_at(380)                                   # a stormy beach, no text
captioned = plain.copy()
cv2.putText(captioned, "SILVER HARBOR", (40, 140), cv2.FONT_HERSHEY_SIMPLEX, 1.3, (255, 255, 255), 3, cv2.LINE_AA)

q_caption = embed([text_part("SILVER HARBOR")])
sim_plain = float(embed([image_part(plain)]) @ q_caption)
sim_captioned = float(embed([image_part(captioned)]) @ q_caption)
best_real = float((F @ q_caption).max())

display(HTML(CSS + "<div class='vr'>"
    f"<div class='vc'><img src='data:image/jpeg;base64,{jpeg_b64(plain, 70)}'><div>without caption: cosine {sim_plain:.3f}</div></div>"
    f"<div class='vc'><img src='data:image/jpeg;base64,{jpeg_b64(captioned, 70)}'><div>caption burned in: cosine {sim_captioned:.3f}</div></div></div>"
    f"<p style='font:13px sans-serif'>Query “SILVER HARBOR”. The best of the {len(F)} real keyframes in the video scores {best_real:.3f}, "
    f"so the captioned frame scores {'higher than every real keyframe' if sim_captioned > best_real else 'close to the best real keyframes'}, although the shot has nothing to do with the words.</p>"))

## 9. The embeddings with their time ranges, and a similarity timeline

The table lists the best clips with the first dimensions of their embedding, the vector norm (1.0 after normalisation) and the cosine score to the text query. The timeline below shows the query's similarity for **every** clip in order, so you can see where in the video the content matches; the best `TOP_K` clips are outlined.

In [ ]:
sims = E @ q_text
order = np.argsort(-sims)[:8]
rows = "".join(
    f"<tr><td class='num'>{r}</td><td>{mmss(meta[r]['startSec'])} – {mmss(meta[r]['endSec'])}</td>"
    f"<td class='num'>[{', '.join(f'{v:+.3f}' for v in E[r, :6])}, …]</td>"
    f"<td class='num'>{np.linalg.norm(E[r]):.2f}</td><td class='num'>{sims[r]:.3f}</td></tr>" for r in order)
display(HTML(CSS + f"<table class='t'><tr><th>row</th><th>time range</th><th>embedding (first 6 of {E.shape[1]} dims)</th>"
                   f"<th>norm</th><th>cosine to “{_html.escape(text_query)}”</th></tr>{rows}</table>"))

# similarity timeline as inline SVG: one bar per clip, height = cosine similarity to the text query
W, H, pad = 900, 140, 24
lo, hi = float(sims.min()), float(sims.max())
bw = (W - 2 * pad) / len(sims)
top = {h["row"] for h in text_hits}
bars = ""
for r, s in enumerate(sims):
    h = 6 + (s - lo) / (hi - lo + 1e-9) * (H - 2 * pad - 6)
    x = pad + r * bw
    bars += (f"<rect x='{x:.1f}' y='{H - pad - h:.1f}' width='{max(bw - 1, 1):.1f}' height='{h:.1f}' "
             f"fill='{'#fb8500' if r in top else '#8ecae6'}'><title>{mmss(meta[r]['startSec'])} cos {s:.3f}</title></rect>")
ticks = "".join(f"<text x='{pad + i * (W - 2 * pad) / 6:.0f}' y='{H - 6}' font-size='11' font-family='sans-serif' text-anchor='{'end' if i == 6 else 'start'}'>"
                f"{mmss(i * end_sec / 6)}</text>" for i in range(7))
display(HTML(f"<svg width='{W}' height='{H}' style='border:1px solid #d0d7de;border-radius:8px;background:#fff'>{bars}{ticks}</svg>"))

In [ ]:
total = sum(dt for dt, _ in timings.values())
rows = "".join(f"<tr><td>{s}</td><td class='num'>{dt:.2f} s</td><td class='num'>{dt / total:.0%}</td><td>{_html.escape(d)}</td></tr>"
               for s, (dt, d) in timings.items())
display(HTML(CSS + f"<table class='t'><tr><th>step</th><th>time</th><th>share</th><th>detail</th></tr>{rows}"
                   f"<tr><th>total</th><th class='num'>{total:.2f} s</th><th></th><th></th></tr></table>"))

## How the flow works

1. **Segment** – the video is cut into fixed clips; each keeps `startSec` and `endSec`. The segmentation is simple: fixed length, no scene detection.
2. **Embed** – each clip (downscaled, 2 fps, a few hundred KB at most) is sent to Qwen3-VL-Embedding as a `video_url`; a few still frames of each clip are embedded as images. The 4096-number vector summarises what is visible in the clip. This model looks at **frames only**: it can read large on-screen text (hard captions), but it does not hear audio, so spoken dialogue, transcripts and music are not searchable.
3. **Index** – the vectors go into two FAISS indexes (clips and keyframes). **Row *i* of the clip index = `segments[i]` = a time range**; each keyframe row points back to its clip, which is the embedding ↔ timestamp mapping. The index and metadata are saved to disk.
4. **Query** – a text or an image goes through the same model. Because text, images and video share one embedding space, a sentence can be compared directly to a clip. A text query is matched against clip vectors and an image query against keyframe vectors, because each pairing works best in practice.
5. **Search** – FAISS returns the nearest clips by cosine similarity; each result is a time range with a thumbnail.
6. **Rerank (optional)** – a vision-language reranker (Qwen3-VL-Reranker-2B) re-scores the top `CANDIDATES` clips through their keyframes and reorders them. It helps most for specific queries (exact on-screen text, one scene among similar ones) and costs a few seconds.
7. **Inspect** – the table and timeline show the vectors with their time ranges and where in the video the query matches.

**Limits to keep in mind**
- The model's context is 8192 tokens, so a clip must stay short. That is why clips are 5 s at 2 fps and 448 px wide; a longer or sharper clip may be rejected.
- Fixed 5 s windows can cut a scene in half; shorter windows give finer timestamps and more vectors, longer windows give more context per vector.
- Search quality depends on what is *visible*. Dialogue and sound are not searchable here.
- A clip is the unit of search: a result tells you *when*, not *where in the frame*.

**Things to tune:** `SEGMENT_SEC`, `CLIP_WIDTH`/`CLIP_FPS` (detail vs. tokens), `WORKERS` (parallel requests to the server), `CANDIDATES` (how many clips the reranker re-scores: more recall, more time), `ANALYZE_SEC` (embed only the start of a long video) and `TOP_K`.